# Noteblock afin de changer le tempo d'un fichier NBS, ainsi que de faire un changement d'instrument simple

In [1]:
import pandas as pd
import math
from ReadNBS import read_file_complet,WriteNBS
from IPython.display import Image

!["titer"](delays.png)

In [2]:
16.93/4

4.2325

In [3]:
1/0.60*4

6.666666666666667

In [4]:
1/0.70*4*2

11.428571428571429

In [5]:
1/1/0.55*4

7.2727272727272725

In [60]:
file_in = "in/abys.nbs"
file_out = 'in/abys3.nbs'

changement_tempo = True
changement_instr = False

tick_second = 1/0.70*4*2#1/1/0.55*4 # Valeur affiché dans noteblock studio


In [102]:
header, data, fin = read_file_complet(file_in)
data["octave"] = data.key//12
data["note"]  = data.key%12
data['real tick'] = data['tick']

Loading file:  in/abys.nbs


In [74]:
if(changement_tempo):
    tick_multiplier = 20/tick_second
else:
    tick_multiplier = 1
    
    
data['real tick'] = [math.ceil(a) for a in data['tick']*tick_multiplier]

In [9]:
if(changement_instr):
    # Si note trop basse, mettre du double basse
    data.loc[data['key'] < 0, 'insts'] = 1
    data.loc[data['key'] < 0, 'key'] += 24

    # Si toujours trop basse, remonter la note
    data.loc[data['key'] < 0, 'key'] += 12
    data.loc[data['key'] < 0, 'key'] += 12

    # Si note trop haute, mettre flute
    data.loc[data['key'] > 24, 'insts'] = 6
    data.loc[data['key'] > 24, 'key'] -= 12

    # Si toujours trop haute, mettre Bell
    data.loc[data['key'] > 24, 'insts'] = 7
    data.loc[data['key'] > 24, 'key'] -= 12

    # Si toujours trop haute, baisser note
    data.loc[data['key'] > 24, 'key'] -= 12
    data.loc[data['key'] > 24, 'key'] -= 12

In [10]:
WriteNBS(data,file_out, header, fin)

In [11]:
data.head()

,tick,layer,key,insts,vels,pans,pits,real tick
0,0,0,11,0,80,100,0,0
1,0,1,15,0,80,100,0,0
2,0,2,11,1,80,100,0,0
3,1,0,22,0,80,100,0,2
4,4,0,18,1,80,100,0,7


In [13]:
data.head()

,tick,layer,key,insts,vels,pans,pits,real tick,octave,note
0,0,0,11,0,80,100,0,0,0,11
1,0,1,15,0,80,100,0,0,1,3
2,0,2,11,1,80,100,0,0,0,11
3,1,0,22,0,80,100,0,2,1,10
4,4,0,18,1,80,100,0,7,1,6


In [14]:
modifier = {-2:{"bass":True, "didgeridoo": False},
            -1:{"bass":True, "didgeridoo": False, "guitar": True},
             0:{"guitar": True, "harp": True},
           }

In [62]:
instruments = {'didgeridoo': 1, 'bass': 1, 'guitar': 2, 'banjo': 3, 'pling': 3, 'iron_xylophone': 3,
               'bit': 3, 'harp': 3, 'cow_bell': 4, 'flute': 4, 'chime': 5, 'xylophone': 5, 'bell': 5}


In [16]:
data = data.sort_values(by=['tick', 'layer'])

In [17]:
data

,tick,layer,key,insts,vels,pans,pits,real tick,octave,note
0,0,0,11,0,80,100,0,0,0,11
1,0,1,15,0,80,100,0,0,1,3
2,0,2,11,1,80,100,0,0,0,11
3,1,0,22,0,80,100,0,2,1,10
4,4,0,18,1,80,100,0,7,1,6
...,...,...,...,...,...,...,...,...,...,...
575,1160,0,5,7,80,100,0,2030,0,5
576,1162,0,1,7,80,100,0,2034,0,1
577,1164,0,22,0,80,100,0,2037,1,10
578,1166,0,1,7,80,100,0,2041,0,1


In [130]:
modifier = np.zeros((5, 13))
modifier[0,1] = 1
modifier[1,1] = 1
modifier[2,1] = 1
modifier[3,1] = 1
modifier[4,1] = 1


modifier[0,0] = 1
modifier[1,0] = 1
modifier[2,0] = 1
modifier[3,0] = 1
modifier[4,0] = 1

In [131]:
modifier

array([[1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

tick = 0
layer = 0
new_rows = [] 


for i in range(data.shape[0]):
    
    if(data.iloc[i].tick > tick):
        tick = data.iloc[i].tick
        layer = 0
    

    
    for instr_i in range(13):
        note = data.iloc[1].copy()
        octave = note.octave
        if(octave<0):
            octave = 0
        if(octave>4):
            octave = 4
    
        if(modifier[octave,instr_i]):
            note.instrum = instr_i
            instr_octave = instruments[list(instruments.keys())[instr_i]]
            if(instr_octave<octave):
                note.key  = note.note + 12
            elif(instr_octave == octave):
                note.key  = note.note
            elif(instr_octave+1 == octave):
                note.key  = note.note + 12
            else:
                note.key  = note.note
            note.layer = layer
            layer += 1
            new_rows.append(note)

    if(data.iloc[i].layer > layer):
        data.iloc[i].layer = layer
    layer += 1

    
data.append(new_rows)

In [132]:
tick = 0
layer = 0
for i in range(data.shape[0]):
    if(data.iloc[i].tick > tick):
        tick = data.iloc[i].tick
        layer = 0
    if(data.iloc[i].layer > layer):
        data.iloc[i].layer = layer
    layer += 1
    
nbs_instruments = {'didgeridoo': 12, 'bass': 1, 'guitar': 5, 'banjo': 14, 'pling': 15, 'iron_xylophone': 10,
               'bit': 13, 'harp': 0, 'cow_bell': 11, 'flute': 6, 'chime': 8, 'xylophone': 9, 'bell': 7}
    
octave_instruments = {'didgeridoo': 1, 'bass': 1, 'guitar': 2, 'banjo': 3, 'pling': 3, 'iron_xylophone': 3,
               'bit': 3, 'harp': 3, 'cow_bell': 4, 'flute': 4, 'chime': 5, 'xylophone': 5, 'bell': 5}
    
    
    
tick = 0
layer = 0
new_rows = []

for i in range(data.shape[0]):
    if data.iloc[i]['tick'] > tick:
        tick = data.iloc[i]['tick']
        layer = 0

    for instr_i in range(13):
        note = data.iloc[i].copy()
        octave = note['octave']
        note['octave'] = max(0, min(octave, 4))  # Ensuring octave is within [0, 4]

        if modifier[octave, instr_i]:
            note['insts'] = instr_i
            instr_octave = nbs_instruments[list(octave_instruments.keys())[instr_i]]
            if instr_octave < octave:
                note['key'] = note['note'] + 12
            elif instr_octave + 1 == octave:
                note['key'] = note['note'] + 12
            else:
                note['key'] = note['note']
            
            note['layer'] = layer
            layer += 1
            new_rows.append(note)

    if data.iloc[i]['layer'] > layer:
        data.at[i, 'layer'] = layer
    elif(data.iloc[i]['layer'] > layer):
        data.at[i, 'layer'] = layer
    layer += 1

#data = pd.concat([data, pd.DataFrame(new_rows)], ignore_index=True)
new_data = pd.DataFrame(new_rows)



tick = 0
layer = 0
for i in range(new_data.shape[0]):
    if(new_data.iloc[i].tick > tick):
        tick = new_data.iloc[i].tick
        layer = 0
    if(new_data.iloc[i].layer > layer):
        new_data.iloc[i].layer = layer
    layer += 1

C:\Users\marce\AppData\Local\Temp\ipykernel_17292\2245207453.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.iloc[i].layer = layer


In [133]:
data

,tick,layer,key,insts,vels,pans,pits,octave,note,real tick
0,0,0,11,0,80,100,0,0,11,0
1,0,1,15,0,80,100,0,1,3,0
2,0,2,11,1,80,100,0,0,11,0
3,1,0,22,0,80,100,0,1,10,1
4,4,0,18,1,80,100,0,1,6,4
...,...,...,...,...,...,...,...,...,...,...
575,1160,0,5,7,80,100,0,0,5,1160
576,1162,0,1,7,80,100,0,0,1,1162
577,1164,0,22,0,80,100,0,1,10,1164
578,1166,0,1,7,80,100,0,0,1,1166


In [134]:
WriteNBS(new_data,file_out, header, fin)

In [51]:
type(data.iloc[1].copy())

pandas.core.series.Series

In [98]:
modifier

array([[0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [58]:
if(modifier[octave, instr_i]):
    print('test')

In [119]:
minecaft_instruments = ['harp',
 'bass',
 'Bass Drum',
 'Snare Drum',
 'Click',
 'Guitar',
 'BFlute',
 'Bell',
 'Chime',
 'Xylophone',
 'Iron Xylophone',
 'Cow Bel',
 'Didgeridoo',
 'Bit',
 'Banjo',
 'Pling']

In [121]:
for i in range(len(minecaft_instruments)):
    print(str(i) + '  ' + minecaft_instruments[i])

0  harp
1  bass
2  Bass Drum
3  Snare Drum
4  Click
5  Guitar
6  BFlute
7  Bell
8  Chime
9  Xylophone
10  Iron Xylophone
11  Cow Bel
12  Didgeridoo
13  Bit
14  Banjo
15  Pling


In [29]:
import numpy as np

In [25]:
import ipywidgets as widgets
from IPython.display import display

In [27]:
# List of items for the checkboxes
items = ['Item 1', 'Item 2', 'Item 3']

# Create a list of checkboxes
checkboxes = [widgets.Checkbox(value=False, description=item) for item in items]

# Display the checkboxes
for checkbox in checkboxes:
    display(checkbox)

# Function to get the values of the checkboxes
def get_checkbox_values(checkboxes):
    return [checkbox.value for checkbox in checkboxes]

# Button to get and display checkbox values
button = widgets.Button(description="Get Values")
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        values = get_checkbox_values(checkboxes)
        print(values)

button.on_click(on_button_clicked)

display(button, output)

Checkbox(value=False, description='Item 1')

Checkbox(value=False, description='Item 2')

Checkbox(value=False, description='Item 3')

Button(description='Get Values', style=ButtonStyle())

Output()

In [30]:
import ipywidgets as widgets
from IPython.display import display

# Create checkboxes only for the yellow cells
checkboxes = {
    (1, 'bass'): widgets.Checkbox(value=False, description=''),
    (2, 'bass'): widgets.Checkbox(value=False, description=''),
    (2, 'iron_xylophone'): widgets.Checkbox(value=False, description=''),
    (2, 'chime'): widgets.Checkbox(value=False, description=''),
    (3, 'iron_xylophone'): widgets.Checkbox(value=False, description=''),
    (3, 'chime'): widgets.Checkbox(value=False, description=''),
    (4, 'guitar'): widgets.Checkbox(value=False, description=''),
    (5, 'guitar'): widgets.Checkbox(value=False, description=''),
    (5, 'harp'): widgets.Checkbox(value=False, description=''),
    (5, 'xylophone'): widgets.Checkbox(value=False, description=''),
}

# List of instruments
instruments = ['didgeridoo', 'bass', 'guitar', 'banjo', 'pling', 'iron_xylophone', 'bit', 'harp', 'cow_bell', 'flute', 'chime', 'xylophone', 'bell']

# Create the grid layout using GridBox
layout = widgets.Layout(
    grid_template_columns='auto ' * (len(instruments) + 1),
    grid_template_rows='auto ' * 6,
    grid_gap='5px'
)

# Create the header row
header = [widgets.Label('Octaves')] + [widgets.Label(inst) for inst in instruments]

# Create the rows with checkboxes
rows = []
for row in range(1, 6):
    row_widgets = [widgets.Label(str(row))]
    for instrument in instruments:
        if (row, instrument) in checkboxes:
            row_widgets.append(checkboxes[(row, instrument)])
        else:
            row_widgets.append(widgets.Label(''))
    rows.append(row_widgets)

# Combine header and rows
all_widgets = header + [item for sublist in rows for item in sublist]

# Display the grid
gridbox = widgets.GridBox(children=all_widgets, layout=layout)
display(gridbox)


GridBox(children=(Label(value='Octaves'), Label(value='didgeridoo'), Label(value='bass'), Label(value='guitar'…

In [31]:
import sys
from PyQt5.QtWidgets import QApplication, QMainWindow, QWidget, QGridLayout, QVBoxLayout, QLabel, QCheckBox

class CheckboxGridWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Checkbox Grid")
        
        # Central widget
        central_widget = QWidget(self)
        self.setCentralWidget(central_widget)
        
        # Create a grid layout
        grid_layout = QGridLayout()
        
        # List of instruments
        instruments = ['didgeridoo', 'bass', 'guitar', 'banjo', 'pling', 'iron_xylophone', 'bit', 'harp', 'cow_bell', 'flute', 'chime', 'xylophone', 'bell']
        
        # Create the header row
        grid_layout.addWidget(QLabel('Octaves'), 0, 0)
        for col, instrument in enumerate(instruments, start=1):
            grid_layout.addWidget(QLabel(instrument), 0, col)
        
        # Define the yellow cells
        yellow_cells = {
            (1, 'bass'),
            (2, 'bass'), (2, 'iron_xylophone'), (2, 'chime'),
            (3, 'iron_xylophone'), (3, 'chime'),
            (4, 'guitar'),
            (5, 'guitar'), (5, 'harp'), (5, 'xylophone'),
        }
        
        # Add the rows with checkboxes
        for row in range(1, 6):
            grid_layout.addWidget(QLabel(str(row)), row, 0)
            for col, instrument in enumerate(instruments, start=1):
                if (row, instrument) in yellow_cells:
                    grid_layout.addWidget(QCheckBox(), row, col)
                else:
                    grid_layout.addWidget(QLabel(''), row, col)
        
        # Set the layout for the central widget
        central_widget.setLayout(grid_layout)

def main():
    app = QApplication(sys.argv)
    window = CheckboxGridWindow()
    window.show()
    sys.exit(app.exec_())

if __name__ == "__main__":
    main()


SystemExit: 0

D:\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [16]:
import sys
from PyQt5.QtWidgets import QApplication, QMainWindow, QWidget, QGridLayout, QLabel, QCheckBox, QStylePainter, QStyleOptionButton, QStyle
from PyQt5.QtGui import QColor, QPalette
from PyQt5.QtCore import Qt

class CustomCheckBox(QCheckBox):
    def __init__(self, color, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.color = color

    def paintEvent(self, event):
        painter = QStylePainter(self)
        opt = QStyleOptionButton()
        self.initStyleOption(opt)
        
        if self.color:
            opt.palette.setColor(QPalette.Button, QColor(self.color))
            painter.drawControl(QStyle.CE_CheckBox, opt)
        else:
            painter.drawControl(QStyle.CE_CheckBox, opt)

class CheckboxGridWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Checkbox Grid")
        
        # Central widget
        central_widget = QWidget(self)
        self.setCentralWidget(central_widget)
        
        # Create a grid layout
        grid_layout = QGridLayout()
        
        # List of instruments
        instruments = ['didgeridoo', 'bass', 'guitar', 'banjo', 'pling', 'iron_xylophone', 
                       'bit', 'harp', 'cow_bell', 'flute', 'chime', 'xylophone', 'bell']
        
        # Create the header row
        grid_layout.addWidget(QLabel('Octaves'), 0, 0)
        for col, instrument in enumerate(instruments, start=1):
            grid_layout.addWidget(QLabel(instrument), 0, col)
        
        # Define the yellow cells
        yellow_cells = {
            (1, 'bass'),
            (2, 'bass'), (2, 'iron_xylophone'), (2, 'chime'),
            (3, 'iron_xylophone'), (3, 'chime'),
            (4, 'guitar'),
            (5, 'guitar'), (5, 'harp'), (5, 'xylophone'),
        }
        
        # Add the rows with checkboxes and color them
        for row in range(1, 6):
            grid_layout.addWidget(QLabel(str(row)), row, 0)
            for col, instrument in enumerate(instruments, start=1):
                color = "yellow" if (row, instrument) in yellow_cells else "gray"
                checkbox = CustomCheckBox(color)
                grid_layout.addWidget(checkbox, row, col)
        
        # Set the layout for the central widget
        central_widget.setLayout(grid_layout)

def main():
    app = QApplication(sys.argv)
    window = CheckboxGridWindow()
    window.show()
    sys.exit(app.exec_())

if __name__ == "__main__":
    main()


SystemExit: 0

In [11]:
instruments = {'didgeridoo':1, 'bass':1, 'guitar':2, 'banjo':3, 'pling':3, 'iron_xylophone':3, 'bit':3, 'harp':3, 
               'cow_bell':4, 'flute':4, 'chime':5, 'xylophone':5, 'bell':5}


In [15]:
instruments

{'didgeridoo': 1,
 'bass': 1,
 'guitar': 2,
 'banjo': 3,
 'pling': 3,
 'iron_xylophone': 3,
 'bit': 3,
 'harp': 3,
 'cow_bell': 4,
 'flute': 4,
 'chime': 5,
 'xylophone': 5,
 'bell': 5}

In [9]:
import sys
from PyQt5.QtWidgets import QApplication, QMainWindow, QWidget, QGridLayout, QLabel, QCheckBox, QFrame
from PyQt5.QtCore import Qt

class CustomCheckBox(QCheckBox):
    def __init__(self, octave, value, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.octave = octave
        self.value = value
        self.setStyleSheet(self.get_stylesheet())

    def get_stylesheet(self):
        if self.octave < self.value:
            return ("QCheckBox::indicator:unchecked { background-color: black; }"
                    "QCheckBox::indicator:checked { background-color: gray; }")
        elif self.octave == self.value or self.octave == self.value + 1:
            return ("QCheckBox::indicator:unchecked { background-color: blue; }"
                    "QCheckBox::indicator:checked { background-color: green; }")
        else:
            return ("QCheckBox::indicator:unchecked { background-color: yellow; }"
                    "QCheckBox::indicator:checked { background-color: orange; }")

class CheckboxGridWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Checkbox Grid")
        
        # Central widget
        central_widget = QWidget(self)
        self.setCentralWidget(central_widget)
        
        # Create a grid layout
        grid_layout = QGridLayout()
        
        # Dictionary of instruments with their values
        instruments = {'didgeridoo': 1, 'bass': 1, 'guitar': 2, 'banjo': 3, 'pling': 3, 'iron_xylophone': 3, 
                       'bit': 3, 'harp': 3, 'cow_bell': 4, 'flute': 4, 'chime': 5, 'xylophone': 5, 'bell': 5}
        
        # Create the header row
        grid_layout.addWidget(QLabel('Octaves'), 0, 0, Qt.AlignCenter)
        for col, instrument in enumerate(instruments.keys(), start=1):
            label = QLabel(instrument)
            grid_layout.addWidget(label, 0, col)
        
        # Add the rows with checkboxes and color them based on conditions
        for row in range(1, 6):
            label = QLabel(str(row))
            grid_layout.addWidget(label, row, 0, Qt.AlignCenter)
            for col, (instrument, value) in enumerate(instruments.items(), start=1):
                checkbox = CustomCheckBox(row, value)
                #checkbox.setStyleSheet(checkbox.get_stylesheet() + " QCheckBox::indicator:checked { image: url(:/qt-project.org/styles/commonstyle/images/checkbox.png); }")
                grid_layout.addWidget(checkbox, row, col, Qt.AlignCenter)
        
        # Set the layout for the central widget
        central_widget.setLayout(grid_layout)

def main():
    app = QApplication(sys.argv)
    window = CheckboxGridWindow()
    window.show()
    sys.exit(app.exec_())

if __name__ == "__main__":
    main()


SystemExit: 0

In [1]:
import sys
from PyQt5.QtWidgets import QApplication, QMainWindow, QWidget, QGridLayout, QLabel, QCheckBox, QFrame
from PyQt5.QtCore import Qt

from PyQt5.QtWidgets import QCheckBox

class CustomCheckBox(QCheckBox):
    def __init__(self, octave, value, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.octave = octave
        self.value = value
        self.setStyleSheet(self.get_stylesheet())

    def get_stylesheet(self):
        cross_style = ("QCheckBox::indicator:checked::before {"
                       "content: '✕';"
                       "color: white;"
                       "font-weight: bold;"
                       "position: absolute;"
                       "top: -2px;"
                       "left: 3px;"
                       "}")

        if self.octave < self.value:
            return ("QCheckBox::indicator { width: 15px; height: 15px; }"
                    "QCheckBox::indicator:unchecked { background-color: black; }"
                    "QCheckBox::indicator:checked { background-color: gray; }"
                    + cross_style)
        elif self.octave == self.value or self.octave == self.value + 1:
            return ("QCheckBox::indicator { width: 15px; height: 15px; }"
                    "QCheckBox::indicator:unchecked { background-color: blue; }"
                    "QCheckBox::indicator:checked { background-color: green; }"
                    + cross_style)
        else:
            return ("QCheckBox::indicator { width: 15px; height: 15px; }"
                    "QCheckBox::indicator:unchecked { background-color: yellow; }"
                    "QCheckBox::indicator:checked { background-color: orange; }"
                    + cross_style)


class CheckboxGridWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Checkbox Grid")
        
        # Central widget
        central_widget = QWidget(self)
        self.setCentralWidget(central_widget)
        
        # Create a grid layout
        grid_layout = QGridLayout()
        
        # Dictionary of instruments with their values
        instruments = {'didgeridoo': 1, 'bass': 1, 'guitar': 2, 'banjo': 3, 'pling': 3, 'iron_xylophone': 3, 
                       'bit': 3, 'harp': 3, 'cow_bell': 4, 'flute': 4, 'chime': 5, 'xylophone': 5, 'bell': 5}
        
        # Create the header row
        grid_layout.addWidget(QLabel('Octaves'), 0, 0, Qt.AlignCenter)
        for col, instrument in enumerate(instruments.keys(), start=1):
            label = QLabel(instrument)
            grid_layout.addWidget(label, 0, col)
        
        # Add the rows with checkboxes and color them based on conditions
        for row in range(1, 6):
            label = QLabel(str(row))
            grid_layout.addWidget(label, row, 0, Qt.AlignCenter)
            for col, (instrument, value) in enumerate(instruments.items(), start=1):
                checkbox = CustomCheckBox(row, value)
                #checkbox.setStyleSheet(checkbox.get_stylesheet() + " QCheckBox::indicator:checked { image: url(:/qt-project.org/styles/commonstyle/images/checkbox.png); }")
                grid_layout.addWidget(checkbox, row, col, Qt.AlignCenter)
        
        # Set the layout for the central widget
        central_widget.setLayout(grid_layout)

def main():
    app = QApplication(sys.argv)
    window = CheckboxGridWindow()
    window.show()
    sys.exit(app.exec_())

if __name__ == "__main__":
    main()


SystemExit: 0

D:\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [138]:
import sys
from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QGridLayout, QLabel, QPushButton,
                             QFileDialog, QVBoxLayout, QCheckBox,QLineEdit)
from PyQt5.QtCore import Qt

class CustomCheckBox(QCheckBox):
    def __init__(self, octave, value, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.octave = octave
        self.value = value
        self.setStyleSheet(self.get_stylesheet())

    def get_stylesheet(self):
        cross_style = ("QCheckBox::indicator:checked::before {"
                       "content: '✕';"
                       "color: white;"
                       "font-weight: bold;"
                       "position: absolute;"
                       "top: -2px;"
                       "left: 3px;"
                       "}")

        if self.octave < self.value:
            return ("QCheckBox::indicator { width: 15px; height: 15px; }"
                    "QCheckBox::indicator:unchecked { background-color: black; }"
                    "QCheckBox::indicator:checked { background-color: gray; }"
                    + cross_style)
        elif self.octave == self.value or self.octave == self.value + 1:
            return ("QCheckBox::indicator { width: 15px; height: 15px; }"
                    "QCheckBox::indicator:unchecked { background-color: blue; }"
                    "QCheckBox::indicator:checked { background-color: green; }"
                    + cross_style)
        else:
            return ("QCheckBox::indicator { width: 15px; height: 15px; }"
                    "QCheckBox::indicator:unchecked { background-color: yellow; }"
                    "QCheckBox::indicator:checked { background-color: orange; }"
                    + cross_style)

class CheckboxGridWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Checkbox Grid")
        
        # Central widget
        central_widget = QWidget(self)
        self.setCentralWidget(central_widget)
        
        # Vertical layout to hold the entire UI
        v_layout = QVBoxLayout()
        central_widget.setLayout(v_layout)

        # Button to load a file
        self.load_file_btn = QPushButton("Load File")
        self.load_file_btn.clicked.connect(self.load_file)
        v_layout.addWidget(self.load_file_btn)

        # Label to display file name
        self.file_label = QLineEdit("No file loaded")
        v_layout.addWidget(self.file_label)

        # Create a grid layout for checkboxes
        grid_layout = QGridLayout()
        v_layout.addLayout(grid_layout)

        # Dictionary of instruments with their values
        instruments = {'didgeridoo': 1, 'bass': 1, 'guitar': 2, 'banjo': 3, 'pling': 3, 'iron_xylophone': 3,
                       'bit': 3, 'harp': 3, 'cow_bell': 4, 'flute': 4, 'chime': 5, 'xylophone': 5, 'bell': 5}

        # Create the header row
        grid_layout.addWidget(QLabel('Octaves'), 0, 0, Qt.AlignCenter)
        for col, instrument in enumerate(instruments.keys(), start=1):
            label = QLabel(instrument)
            grid_layout.addWidget(label, 0, col)

        # Add the rows with checkboxes
        self.checkboxes = []
        for row in range(1, 6):
            label = QLabel(str(row))
            grid_layout.addWidget(label, row, 0, Qt.AlignCenter)
            row_checkboxes = []
            for col, (instrument, value) in enumerate(instruments.items(), start=1):
                checkbox = CustomCheckBox(row, value)
                grid_layout.addWidget(checkbox, row, col, Qt.AlignCenter)
                row_checkboxes.append(checkbox)
            self.checkboxes.append(row_checkboxes)

        # Save button
        self.save_btn = QPushButton("Save")
        self.save_btn.clicked.connect(self.save_data)
        v_layout.addWidget(self.save_btn)

    def load_file(self):
        options = QFileDialog.Options()
        file_name, _ = QFileDialog.getOpenFileName(self, "Open File", "", "All Files (*)", options=options)
        if file_name:
            self.file_label.setText(file_name.split('/')[-1])

    def save_data(self):
        # Initialize an array to hold the state of each checkbox
        checkbox_states = []

        # Loop through each row of checkboxes
        for row_checkboxes in self.checkboxes:
            row_states = []
            # Loop through each checkbox in the row
            for checkbox in row_checkboxes:
                # Append the state of the checkbox (True if checked, False if not)
                row_states.append(checkbox.isChecked())
            # Append the states of the current row to the main array
            checkbox_states.append(row_states)

        # Optionally, print the array to console to verify (you can remove this later)
        for row_state in checkbox_states:
            print(np.array(row_state))

        # Here you can add additional code to save `checkbox_states` to a file or process it further


def main():
    app = QApplication(sys.argv)
    window = CheckboxGridWindow()
    window.show()
    sys.exit(app.exec_())

if __name__ == "__main__":
    main()


[False False False False False False False False False False False False
 False]
[False False False False False False False False False False False False
 False]
[False False False False False False False False False False False False
 False]
[False False False False  True False False False False False False False
 False]
[False False False False False False False False False False False False
 False]


SystemExit: 0

In [139]:
nbs_instruments = {'didgeridoo': 12, 'bass': 1, 'guitar': 5, 'banjo': 14, 'pling': 15, 'iron_xylophone': 10,
                       'bit': 13, 'harp': 0, 'cow_bell': 11, 'flute': 6, 'chime': 8, 'xylophone': 9, 'bell': 7}

octave_instruments = {'didgeridoo': 1, 'bass': 1, 'guitar': 2, 'banjo': 3, 'pling': 3, 'iron_xylophone': 3,
                       'bit': 3, 'harp': 3, 'cow_bell': 4, 'flute': 4, 'chime': 5, 'xylophone': 5, 'bell': 5}

In [144]:
nbs_instruments[list(octave_instruments.keys())[0]] 

12